In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from preclean import preprocessing
import optuna
from sklearn.model_selection import train_test_split
from optuna.integration import XGBoostPruningCallback

In [ ]:
# ── Uploading train ─────────────────────────────────
df_transac = pd.read_csv('data/train_transaction.csv')
df_id      = pd.read_csv('data/train_identity.csv')
df_train   = preprocessing(df_transac, df_id)

X = df_train.drop(columns=['TransactionID', 'isFraud'])
y = df_train['isFraud']

# ── Uploading test Kaggle ───────────────────────────
df_test_t  = pd.read_csv('data/test_transaction.csv')
df_test_i  = pd.read_csv('data/test_identity.csv')
df_test    = preprocessing(df_test_t, df_test_i)

X_test = df_test.drop(columns=['TransactionID'])

# Aligning columns (because of the get_dummies)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

Choice of parameters with Optuna

In [ ]:
X_sample, _, y_sample, _ = train_test_split(
    X, y, train_size=0.2, stratify=y, random_state=42
)
print(f"Échantillon : {len(X_sample)} lignes au lieu de {len(X)}")

In [ ]:
def objective(trial):
    params = {
        'max_depth':        trial.suggest_int('max_depth', 3, 9), #under 3 -> under fitting, above 9 -> over fitting
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True), #standard to be sure of the convergence
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0), #the number of data per tree
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20), #constraint on each tree
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),

        # Fixed parameters
        'n_estimators':          300,
        'scale_pos_weight':      len(y[y==0]) / len(y[y==1]),
        'tree_method':           'hist',
        'eval_metric':           'auc',
        'early_stopping_rounds': 30,
        'random_state':          42,
        'n_jobs':                -1,
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []

    for idx_train, idx_val in skf.split(X_sample, y_sample):
        X_tr, X_val = X.iloc[idx_train], X.iloc[idx_val]
        y_tr, y_val = y.iloc[idx_train], y.iloc[idx_val]

        pruning_callback = XGBoostPruningCallback(trial, 'validation_0-auc')

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        preds = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))

    return np.mean(scores)



study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

In [ ]:
import optuna.visualization.matplotlib as optuna_mpl
import matplotlib.pyplot as plt

optuna_mpl.plot_optimization_history(study)
plt.show()

The Optuna optimization converged on the 6th out of 20 attempts, confirming that the chosen search space was sufficiently targeted.

In [ ]:
print("Meilleurs paramètres :", study.best_params)
print("Meilleure AUC        :", study.best_value)

In [ ]:
best_params = study.best_params.copy()

best_params.update({
    'n_estimators':          1000,
    'scale_pos_weight':      len(y[y==0]) / len(y[y==1]),
    'tree_method':           'hist',
    'eval_metric':           'auc',
    'early_stopping_rounds': 50,
    'random_state':          42,
    'n_jobs':                -1,
})

print("Final parameters used :")
for k, v in best_params.items():
    print(f"  {k}: {v}")

In [ ]:
skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof    = np.zeros(len(X))
scores = []

for fold, (idx_train, idx_val) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[idx_train], X.iloc[idx_val]
    y_tr, y_val = y.iloc[idx_train], y.iloc[idx_val]

    model = XGBClassifier(**best_params)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=100
    )

    oof[idx_val] = model.predict_proba(X_val)[:, 1]
    scores.append(roc_auc_score(y_val, oof[idx_val]))
    print(f"Fold {fold+1} AUC : {scores[-1]:.4f}")

print(f"\nAUC moyenne    : {np.mean(scores):.4f}")
print(f"AUC OOF totale : {roc_auc_score(y, oof):.4f}")

In [ ]:
skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof    = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
scores = []

for fold, (idx_train, idx_val) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[idx_train], X.iloc[idx_val]
    y_tr, y_val = y.iloc[idx_train], y.iloc[idx_val]

    model = XGBClassifier(**best_params)

    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)

    oof[idx_val] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

    scores.append(roc_auc_score(y_val, oof[idx_val]))
    print(f"Fold {fold+1} AUC : {scores[-1]:.4f}")

print(f"MeanAUC: {np.mean(scores):.4f}")
print(f"AUC OOF totale : {roc_auc_score(y, oof):.4f}")

In [ ]:
submission = pd.DataFrame({
    'TransactionID': df_test['TransactionID'],
    'isFraud': test_preds
})
submission.to_csv('submission.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import xgboost as xgb

fig, ax = plt.subplots(figsize=(10, 8))
xgb.plot_importance(model, max_num_features=20, ax=ax, importance_type='gain')
plt.title('Most important variables for fraud detection: Top 20')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()